# Day 8: Slicing, masks, saturation limits, and fractional flow

**Time:** about 1 hour  |  **Needs:** Days 1 to 7
**Reservoir idea:** saturation limits (connate water, residual oil) and the **fractional-flow curve**, which says what share of the flowing fluid is water at each saturation
**Python you will learn:** NumPy arrays II: slicing (`a[2:5]`, `a[::-1]`, `a[1:] - a[:-1]`), boolean masks (selecting items by a condition), `np.where`, `np.clip`, and figures with several panels (`plt.subplots`)

By the end of today you will be able to:

1. cut an array into pieces with slices, and read it backwards,
2. select, count and replace the items that satisfy a condition, without a loop,
3. keep saturations inside their physical limits with `np.clip`,
4. calculate the fractional flow of water from the relative permeabilities and the viscosities,
5. draw several plots side by side and explain how oil viscosity changes the waterflood.

## 0. Setup and recap

Run the next cell. It also writes the laboratory table of Day 7 again, so this notebook works on its own.

**Recap of Day 7.** An array calculates on all its items at once (`2 * a`, `1 / a`), `np.linspace` makes evenly spaced numbers, `np.loadtxt(..., unpack=True)` loads the columns of a table, `np.interp(x, xs, ys)` interpolates it, and the mobility is $\lambda = k_r/\mu$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

corey = {"swc": 0.2, "sor": 0.2, "krw_max": 0.4, "kro_max": 1.0, "nw": 2.0, "no": 2.0}
MU_W = 1.0     # water viscosity, cp
MU_O = 2.0     # oil viscosity, cp (an illustrative value; we vary it later)

sat_text = """SW KRO KRW PCOW(PSI)
0.000 0.800 0.000 68.0
0.040 0.800 0.000 68.0
0.080 0.800 0.000 68.0
0.120 0.800 0.000 68.0
0.160 0.800 0.000 50.0
0.200 0.650 0.002 32.0
0.240 0.513 0.010 21.0
0.280 0.400 0.020 15.5
0.320 0.315 0.033 12.0
0.360 0.250 0.049 9.20
0.400 0.196 0.066 7.00
0.440 0.150 0.090 5.30
0.480 0.112 0.119 4.20
0.520 0.082 0.150 3.40
0.560 0.060 0.186 2.70
0.600 0.040 0.227 2.10
0.640 0.024 0.277 1.70
0.680 0.012 0.330 1.30
0.720 0.005 0.390 1.00
0.760 0.000 0.462 0.70
0.800 0.000 0.540 0.50
0.840 0.000 0.620 0.40
1.000 0.000 1.000 0.00
"""
with open("sat.txt", "w") as f:
    f.write(sat_text)

sw_t, kro_t, krw_t, pc_t = np.loadtxt("sat.txt", skiprows=1, unpack=True)
print("NumPy", np.__version__, "| table with", len(sw_t), "rows | ready")

## 1. Story: the well that turned to water

*(An invented but typical situation.)* A producer makes clean oil for two years. Then the report shows 5% water, six months later 30%, then 70%, and the well is closed. The engineer who planned the waterflood is asked: "Could we have known?"

The answer is yes. Water and oil do not move as two separate columns. At every saturation the rock lets a *share* of the total flow be water and the rest oil, and that share depends only on the relative permeabilities and the viscosities. This share is the **fractional flow** $f_w$. Today we compute it, plot it, and use it to see how a more viscous oil makes the water arrive sooner. On the way we learn the NumPy tools that this needs: cutting arrays into pieces, selecting items by a condition, and limiting values to a range.

## 2. Slicing arrays

On Day 1 you cut lists with `[start:stop]`. Arrays are cut the same way, and there is one more part: `[start:stop:step]`. Remember that the **start is included and the stop is not**.

| You write | You get (for `a`) |
|---|---|
| `a[2:5]` | items 2, 3, 4 |
| `a[:3]` | the first three items |
| `a[3:]` | everything from item 3 on |
| `a[-3:]` | the last three items |
| `a[::2]` | every second item |
| `a[::-1]` | the whole array **backwards** |

**Predict.** Before you run the next cell, write down on paper what you expect for each of the last four lines, then compare.

In [ ]:
a = np.arange(10, 20)      # 10, 11, ..., 19
print(a)
print(a[2:5])
print(a[-3:])
print(a[::3])
print(a[::-1][:2])

`a[::3]` takes items 0, 3, 6, 9. `a[::-1]` is the array backwards, and `[:2]` then takes its first two items, which are the last two of the original, in reverse: `19, 18`.

**Slices are forgiving.** Asking for one item that does not exist is an error, but a slice that runs past the end just stops at the end:

In [ ]:
print(a[5:100])        # fine: stops at the end
try:
    print(a[100])      # an item that does not exist
except IndexError as error:
    print("IndexError:", error)

**One more thing to know.** A slice of an array is not a copy; it is a **window onto the same numbers**. Changing the slice changes the original. Use `.copy()` when you want a separate array.

In [ ]:
b = np.array([1, 2, 3, 4, 5])
window = b[1:3]
window[0] = 99
print(b)               # b changed too!

c = np.array([1, 2, 3, 4, 5])
separate = c[1:3].copy()
separate[0] = 99
print(c)               # c is untouched

**Your turn 2.1.** Using the saturation column `sw_t` of the laboratory table, store: `first_five` (the first five items), `last_three` (the last three), and `every_fourth` (items 0, 4, 8, ... up to the end).

In [ ]:
# YOUR CODE HERE
first_five = None
last_three = None
every_fourth = None

In [ ]:
# Check (do not edit)
assert np.allclose(first_five, [0.0, 0.04, 0.08, 0.12, 0.16]), f"first_five is {first_five}"
assert np.allclose(last_three, [0.8, 0.84, 1.0]), f"last_three is {last_three}"
assert np.allclose(every_fourth, [0.0, 0.16, 0.32, 0.48, 0.64, 0.80]), f"every_fourth is {every_fourth}"
print("Correct")

<details><summary>Solution</summary>

```python
first_five = sw_t[:5]
last_three = sw_t[-3:]
every_fourth = sw_t[::4]
```

</details>

### Slices that shift: neighbours

A very useful trick: `a[1:]` and `a[:-1]` are the same array shifted by one place. Subtracting them gives **the difference between each item and the next**.

```
a        = [ 0.0  0.04  0.08  0.12 ... ]
a[1:]    = [      0.04  0.08  0.12 ... ]      (drop the first)
a[:-1]   = [ 0.0  0.04  0.08  ...      ]      (drop the last)
a[1:] - a[:-1] = [ 0.04  0.04  0.04 ... ]
```

In a simulator, the flow between two blocks depends on the pressure **difference** between neighbours, so this trick returns on Day 9 (as `np.diff`, which is this same subtraction in one call).

**Your turn 2.2.** The table does not have evenly spaced saturations. Calculate the array `dsw` of the steps between consecutive saturations of `sw_t` (using `sw_t[1:]` and `sw_t[:-1]`). Then store `n_steps` (how many steps there are) and `biggest_step` (the largest one).

In [ ]:
# YOUR CODE HERE
dsw = None
n_steps = None
biggest_step = None

In [ ]:
# Check (do not edit)
assert isinstance(dsw, np.ndarray) and len(dsw) == 22, "dsw should be an array with 22 items (23 points have 22 gaps between them)"
assert n_steps == 22, f"n_steps should be 22, got {n_steps}"
assert abs(biggest_step - 0.16) < 1e-9, f"the biggest step is 0.84 -> 1.00, i.e. 0.16, got {biggest_step}"
print("Correct: 23 points, 22 gaps; the biggest gap is the last one")

<details><summary>Solution</summary>

```python
dsw = sw_t[1:] - sw_t[:-1]
n_steps = len(dsw)
biggest_step = dsw.max()
```
A table with `n` points always has `n - 1` gaps. This is the same idea as fence posts and fence sections.
</details>

### Reading a table backwards

On Day 7 you found the capillary pressure at a given saturation with `np.interp`. The reservoir asks the opposite question: **given a capillary pressure (from the height above the free-water level), what is the saturation?**

`np.interp(x, xs, ys)` needs `xs` to be **increasing**. But `pc_t` *falls* as `sw_t` rises. The cure is to read both arrays backwards with `[::-1]`, which makes the pressures increase:

In [ ]:
print(pc_t[:6])
print(pc_t[::-1][:6])

**Your turn 2.3.** Write `sw_from_pc(pc)` that returns the water saturation for a capillary pressure `pc` (psi), by interpolating with the arrays reversed. Test values to expect: at 32 psi the saturation is exactly 0.2 (it is a table row), and at 10 psi it lies between the rows $(0.32,\,12.0)$ and $(0.36,\,9.2)$.

(Avoid asking for 68 psi: the table repeats that value for four rows, so the answer is not unique.)

In [ ]:
# YOUR CODE HERE
def sw_from_pc(pc):
    """Water saturation at capillary pressure pc (psi), from the laboratory table."""
    pass

In [ ]:
# Check (do not edit)
assert abs(sw_from_pc(32.0) - 0.2) < 1e-9, f"32 psi is the row Sw = 0.2, got {sw_from_pc(32.0)}"
assert abs(sw_from_pc(10.0) - 0.34857) < 1e-4, f"10 psi should give about 0.3486, got {sw_from_pc(10.0)}"
assert abs(sw_from_pc(0.0) - 1.0) < 1e-9, "zero capillary pressure is the free-water level: Sw = 1"
print("Correct: 10 psi corresponds to Sw =", round(float(sw_from_pc(10.0)), 4))

<details><summary>Solution</summary>

```python
def sw_from_pc(pc):
    """Water saturation at capillary pressure pc (psi), from the laboratory table."""
    return np.interp(pc, pc_t[::-1], sw_t[::-1])
```
This is what `invert_decreasing` in `kusim/tables.py` does. Together with Day 7's height equation (pressure = height $\times$ 0.433 $\times$ $(\gamma_w-\gamma_o)$) it gives the initial water saturation at any height.
</details>

## 3. Boolean masks: selecting by a condition

A comparison on an array does not give one answer, it gives **one answer per item**, an array of `True` and `False`:

In [ ]:
mask = sw_t > 0.5
print(sw_t[8:16])
print(mask[8:16])

Such a **mask** can be used as an index: `a[mask]` keeps only the items where the mask is `True`. The mask and the array must have the same length, so a mask made from one column can select from any other column of the table.

| You write | You get |
|---|---|
| `mask = sw_t > 0.5` | the mask (`True` where the condition holds) |
| `sw_t[mask]` | the saturations above 0.5 |
| `krw_t[mask]` | **the water relative permeabilities at those same rows** |
| `mask.sum()` | **how many** items satisfy it (`True` counts as 1) |
| `mask.any()`, `mask.all()` | is **at least one** / are **all** of them `True`? |
| `(a > 1) & (a < 5)` | **and** (both conditions) |
| `(a < 1) \| (a > 5)` | **or** (either) |
| `~mask` | **not** (flips every item) |

In [ ]:
print("rows above Sw = 0.5:", mask.sum(), "of", len(sw_t))
print("krw there:", krw_t[mask][:4], "...")
print("any water at Sw < 0.2?", (krw_t[sw_t < 0.2] > 0).any())

**Careful with `and`.** For masks, you must use `&` and `|` with **parentheses** around each comparison. The words `and` / `or` work on single values (Day 2), not on arrays. See the error, so that you recognise it when you meet it:

In [ ]:
try:
    both = (sw_t > 0.2) and (sw_t < 0.6)
except ValueError as error:
    print("ValueError:", error)

both = (sw_t > 0.2) & (sw_t < 0.6)      # the correct way
print(both.sum(), "rows between 0.2 and 0.6 (exclusive)")

**A warning about `==` on decimals.** Numbers that look equal may differ in the last binary digit, so a mask made with `==` on calculated numbers can miss items. Compare with `np.isclose(a, value)` instead, or use `<` and `>`.

In [ ]:
x = np.linspace(0, 1, 11)
print(x[3] == 0.3, "  <- the item is 0.30000000000000004")
print(np.isclose(x[3], 0.3))

**Your turn 3.1.** In the laboratory table, oil can only flow where $k_{ro} > 0$. Store `n_mobile_oil` (the number of rows with $k_{ro} > 0$) and `sw_last_mobile` (the **largest** saturation among those rows).

In [ ]:
# YOUR CODE HERE
n_mobile_oil = None
sw_last_mobile = None

In [ ]:
# Check (do not edit)
assert n_mobile_oil == 19, f"19 rows have kro > 0, got {n_mobile_oil}"
assert abs(sw_last_mobile - 0.72) < 1e-9, f"the last row with mobile oil is Sw = 0.72, got {sw_last_mobile}"
print("Correct: the oil stops flowing after Sw = 0.72 (the table reaches kro = 0 at 0.76)")

<details><summary>Solution</summary>

```python
oil_flows = kro_t > 0
n_mobile_oil = oil_flows.sum()
sw_last_mobile = sw_t[oil_flows].max()
```

</details>

**Your turn 3.2.** The *transition zone* of Day 7 starts at the connate saturation $S_{wc}=0.2$. Select the rows with $0.2 \le S_w \le 0.6$ and store `n_zone` (how many) and `pc_mean` (the average capillary pressure over those rows). Use `&` with parentheses.

In [ ]:
# YOUR CODE HERE
n_zone = None
pc_mean = None

In [ ]:
# Check (do not edit)
assert n_zone == 11, f"11 rows lie in 0.2..0.6 (both ends included), got {n_zone}"
assert abs(pc_mean - 10.4) < 1e-9, f"the mean of the 11 pressures is 10.4 psi, got {pc_mean}"
print(f"Correct: {n_zone} rows, mean Pc = {pc_mean:.1f} psi")

<details><summary>Solution</summary>

```python
in_zone = (sw_t >= 0.2) & (sw_t <= 0.6)
n_zone = in_zone.sum()
pc_mean = pc_t[in_zone].mean()
```

</details>

### Replacing values with a mask

A mask can also be used on the **left** of `=`, to change only the selected items. Here is a case that every simulator author meets. After a time step, the calculated saturations should lie between $S_{wc}=0.2$ and $1-S_{or}=0.8$, but the explicit update can overshoot a little:

In [ ]:
sw_raw = np.array([0.20, 0.35, 0.8004, 0.79, 1.0003, 0.19999])

too_high = sw_raw > 0.8
print(too_high)

fixed_demo = sw_raw.copy()
fixed_demo[too_high] = 0.8        # replace only the selected items
print(fixed_demo)

**Your turn 3.3.** Work on a copy of `sw_raw` called `sw_fixed`. First store `n_bad`, the number of items **outside** the limits (below 0.2 or above 0.8, using `|`). Then set the items above 0.8 to 0.8 and the items below 0.2 to 0.2 using two masks.

In [ ]:
# YOUR CODE HERE
n_bad = None
sw_fixed = None

In [ ]:
# Check (do not edit)
assert n_bad == 3, f"three items are outside [0.2, 0.8] (0.8004, 1.0003, 0.19999), got {n_bad}"
assert np.allclose(sw_fixed, [0.2, 0.35, 0.8, 0.79, 0.8, 0.2]), f"sw_fixed is {sw_fixed}"
assert sw_raw[4] == 1.0003, "sw_raw must stay unchanged: work on a copy"
print("Correct")

<details><summary>Solution</summary>

```python
n_bad = ((sw_raw < 0.2) | (sw_raw > 0.8)).sum()
sw_fixed = sw_raw.copy()
sw_fixed[sw_fixed > 0.8] = 0.8
sw_fixed[sw_fixed < 0.2] = 0.2
```
Two lines of replacement did the job. The next section does it in one.
</details>

## 4. `np.where` and `np.clip`

### `np.where`: an `if` for a whole array

On Day 2 you wrote `if ... else ...` for one value. `np.where(condition, value_if_true, value_if_false)` does it for **every item** and returns an array:

In [ ]:
rates = np.array([2450, 2410, 2395, 0, 2360, 2330, 2310, 0, 2285, 2260])
print(np.where(rates > 0, "on", "shut-in"))
print(np.where(rates > 0, rates, np.nan))        # hide the zero days: nan means "no value"

### `np.clip`: keep values between limits

`np.clip(a, low, high)` replaces everything below `low` by `low` and everything above `high` by `high`. It is exactly what you did with two masks in 3.3:

In [ ]:
print(np.clip(sw_raw, 0.2, 0.8))

**Your turn 4.1.** Store in `sw_clipped` the array `sw_raw` limited to $[0.2, 0.8]$ with one call, and check by eye that it equals your `sw_fixed` of 3.3.

In [ ]:
# YOUR CODE HERE
sw_clipped = None

In [ ]:
# Check (do not edit)
assert np.allclose(sw_clipped, [0.2, 0.35, 0.8, 0.79, 0.8, 0.2]), f"sw_clipped is {sw_clipped}"
assert np.allclose(sw_clipped, sw_fixed), "it should equal the result of the two masks"
print("Correct: one call instead of two masks")

<details><summary>Solution</summary>

```python
sw_clipped = np.clip(sw_raw, 0.2, 0.8)
```
This line is in the simulator: the explicit saturation update ends with `s_new = np.clip(s_new, m.swc, 1.0 - m.sor)` (see the bridge in Section 9).
</details>

### Saturation limits and the Corey curves

The **connate water** $S_{wc}$ is the water that cannot be moved, and the **residual oil** $S_{or}$ is the oil that a waterflood cannot push out. Between the two, the normalised saturation $s$ goes from 0 to 1. On Day 7 we only evaluated the curves **inside** that range. What happens outside, for instance at $S_w = 0.1$?

In [ ]:
sw_test = 0.1
s = (sw_test - corey["swc"]) / (1 - corey["swc"] - corey["sor"])
print("s =", round(s, 3))
print("krw =", corey["krw_max"] * s ** corey["nw"], " <- water 'flows' although it is below connate!")

A negative $s$ squared becomes positive, so the formula invents a water relative permeability where there is none. (With a non-integer exponent such as 2.5 you would get `nan`, "not a number".) The cure is to **clamp** $s$ to the range 0 to 1 first:

$$s = \mathrm{clip}\!\left(\frac{S_w - S_{wc}}{1 - S_{wc} - S_{or}},\; 0,\; 1\right)$$

Then below $S_{wc}$ we get $k_{rw}=0$ and $k_{ro}=k_{ro}^{\max}$; above $1-S_{or}$ we get $k_{rw}=k_{rw}^{\max}$ and $k_{ro}=0$.

**Your turn 4.2.** Write `corey_krw_kro(sw, props)` where `sw` is **any array** (it may go outside the limits). It returns the arrays `krw, kro`, using the clamped $s$ above and the equations of Day 6 / 7.

In [ ]:
# YOUR CODE HERE
def corey_krw_kro(sw, props):
    """Corey relative permeabilities for any array of saturations, clamped to the limits."""
    pass

In [ ]:
# Check (do not edit)
sw_probe = np.array([0.0, 0.1, 0.2, 0.5, 0.8, 0.9, 1.0])
krw_p, kro_p = corey_krw_kro(sw_probe, corey)
assert np.allclose(krw_p, [0, 0, 0, 0.1, 0.4, 0.4, 0.4]), f"krw is {krw_p}"
assert np.allclose(kro_p, [1, 1, 1, 0.25, 0, 0, 0]), f"kro is {kro_p}"
print("Correct: flat below the connate water and above 1 - Sor")

<details><summary>Solution</summary>

```python
def corey_krw_kro(sw, props):
    """Corey relative permeabilities for any array of saturations, clamped to the limits."""
    s = np.clip((sw - props["swc"]) / (1 - props["swc"] - props["sor"]), 0, 1)
    krw = props["krw_max"] * s ** props["nw"]
    kro = props["kro_max"] * (1 - s) ** props["no"]
    return krw, kro
```

</details>

Plot it over the **whole** range 0 to 1, to see the flat ends:

In [ ]:
sw_full = np.linspace(0, 1, 201)
krw_full, kro_full = corey_krw_kro(sw_full, corey)

plt.figure(figsize=(6, 4))
plt.plot(sw_full, kro_full, color="tab:green", linewidth=2, label="oil, $k_{ro}$")
plt.plot(sw_full, krw_full, color="tab:blue", linewidth=2, label="water, $k_{rw}$")
plt.axvline(corey["swc"], color="gray", linestyle=":")
plt.axvline(1 - corey["sor"], color="gray", linestyle=":")
plt.title("Corey curves with saturation limits (dotted)")
plt.xlabel("water saturation $S_w$ (fraction)")
plt.ylabel("relative permeability (fraction)")
plt.legend()
plt.grid(True)
plt.show()

## 5. Fractional flow

### The equation

In a 1-D bed, with no gravity and no capillary pressure, water and oil feel **the same pressure gradient**. Darcy's law for each phase (units aside) is

$$q_w = -\frac{k\,k_{rw}}{\mu_w}\,A\,\frac{dP}{dx}, \qquad q_o = -\frac{k\,k_{ro}}{\mu_o}\,A\,\frac{dP}{dx}$$

The **fractional flow of water** is the water share of the total flow, $f_w = q_w/(q_w+q_o)$. Everything cancels except the mobilities $\lambda=k_r/\mu$:

$$f_w = \frac{\lambda_w}{\lambda_w+\lambda_o} = \frac{1}{1+\dfrac{k_{ro}\,\mu_w}{k_{rw}\,\mu_o}} \tag{1}$$

What to expect: $f_w=0$ where there is no mobile water ($k_{rw}=0$, at $S_{wc}$), $f_w=1$ where there is no mobile oil ($k_{ro}=0$, at $1-S_{or}$), and an S-shaped rise in between.

| Symbol | Meaning | Name in `kusim` |
|---|---|---|
| $f_w$ | fractional flow of water | `fw` |
| $\lambda_w,\ \lambda_o$ | phase mobilities, $k_r/\mu$ | `lam_w`, `lam_o` |
| $k_{rw},\ k_{ro}$ | relative permeabilities | `krw`, `kro` |
| $\mu_w,\ \mu_o$ | viscosities (cp) | `mu_w`, `mu_o` |

### By hand (15 minutes)

Use the Corey curves ($S_{wc}=S_{or}=0.2$, $k_{rw}^{\max}=0.4$, $n=2$), $\mu_w = 1$ cp and $\mu_o = 2$ cp. The relative permeabilities are given. On paper, calculate the mobilities and $f_w$ with equation (1), then confirm in 5.1.

| $S_w$ | $k_{rw}$ | $k_{ro}$ | $\lambda_w=k_{rw}/\mu_w$ | $\lambda_o=k_{ro}/\mu_o$ | $f_w$ |
|---|---|---|---|---|---|
| 0.4 | 0.0444 | 0.4444 | | | |
| 0.5 | 0.1000 | 0.2500 | | | |
| 0.6 | 0.1778 | 0.1111 | | | |

**Your turn 5.1.** Write `fractional_flow(krw, kro, mu_w, mu_o)` using the first form of equation (1), $\lambda_w/(\lambda_w+\lambda_o)$. It must work on single numbers and on arrays. Compare with your hand calculation (the check cell prints the three values).

In [ ]:
# YOUR CODE HERE
def fractional_flow(krw, kro, mu_w, mu_o):
    """Fractional flow of water from relative permeabilities and viscosities."""
    pass

In [ ]:
# Check (do not edit)
krw3, kro3 = corey_krw_kro(np.array([0.4, 0.5, 0.6]), corey)
fw3 = fractional_flow(krw3, kro3, MU_W, MU_O)
assert np.allclose(fw3, [0.1667, 0.4444, 0.7619], atol=1e-4), f"fw is {fw3}"
assert fractional_flow(0.0, 1.0, 1.0, 2.0) == 0.0, "no mobile water: fw = 0"
assert fractional_flow(0.4, 0.0, 1.0, 2.0) == 1.0, "no mobile oil: fw = 1"
print("Correct:", np.round(fw3, 4), "  (compare with your table)")

<details><summary>Solution</summary>

```python
def fractional_flow(krw, kro, mu_w, mu_o):
    """Fractional flow of water from relative permeabilities and viscosities."""
    lam_w = krw / mu_w
    lam_o = kro / mu_o
    return lam_w / (lam_w + lam_o)
```
We use the form $\lambda_w/(\lambda_w+\lambda_o)$ because it never divides by $k_{rw}$. The second form of equation (1) divides by $k_{rw}$, which is zero at the connate saturation; `kusim` guards that case with `np.where` (see the bridge).
</details>

### The curve

A student's first question is "what does it look like?". We draw $f_w$ for the Corey curves and for the laboratory table. For the table we simply apply the function to its columns, with no loop.

In [ ]:
sw_c = np.linspace(0.2, 0.8, 101)
krw_c, kro_c = corey_krw_kro(sw_c, corey)
fw_c = fractional_flow(krw_c, kro_c, MU_W, MU_O)

fw_t = fractional_flow(krw_t, kro_t, MU_W, MU_O)

plt.figure(figsize=(6, 4))
plt.plot(sw_c, fw_c, color="tab:blue", linewidth=2, label="Corey")
plt.plot(sw_t, fw_t, "o--", color="tab:orange", markersize=4, label="laboratory table")
plt.title("Fractional flow of water ($\\mu_w$ = 1 cp, $\\mu_o$ = 2 cp)")
plt.xlabel("water saturation $S_w$ (fraction)")
plt.ylabel("fractional flow $f_w$ (fraction)")
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.legend()
plt.grid(True)
plt.show()

Look at the table's curve at the left: $f_w=0$ up to $S_w\approx 0.16$ (no mobile water yet), then it rises steeply and reaches 1 near $S_w=0.76$ (no mobile oil left). The Corey curve is the smooth version of the same shape.

At a producer, $f_w$ is the **water cut** (at reservoir conditions). So the story of the well that turned to water is the story of the saturation at the producer climbing along this curve.

**Your turn 5.2.** Using the arrays `sw_c` and `fw_c` above, find the saturation where $f_w$ **first reaches 0.5**: select the saturations where `fw_c >= 0.5` and take the first one. Store it in `sw_half`. (Check against a hand calculation: $f_w=0.5$ when $\lambda_w=\lambda_o$, which for this case is $s = 1/(1+\sqrt{0.8})$.)

In [ ]:
# YOUR CODE HERE
sw_half = None

In [ ]:
# Check (do not edit)
assert abs(sw_half - 0.5167) < 0.006, f"the 50 % point is near Sw = 0.517 (grid step 0.006), got {sw_half}"
print(f"Correct: the flow is half water at Sw = {sw_half:.3f}")

<details><summary>Solution</summary>

```python
sw_half = sw_c[fw_c >= 0.5][0]
```

</details>

## 6. Several plots in one figure

So far every figure had one panel. To compare curves, put the panels side by side with `plt.subplots`. It returns the **figure** and the **axes** (the panels); you then draw on each axes with `ax.plot(...)`.

```python
fig, axes = plt.subplots(1, 2, figsize=(10, 4))   # 1 row, 2 columns
axes[0].plot(...)                                  # left panel
axes[1].plot(...)                                  # right panel
plt.tight_layout()                                 # avoid overlapping labels
plt.show()
```

The commands on an `axes` have slightly different names from the `plt.` commands you know:

| With `plt.` | On an axes `ax` |
|---|---|
| `plt.title("...")` | `ax.set_title("...")` |
| `plt.xlabel("...")` | `ax.set_xlabel("...")` |
| `plt.ylabel("...")` | `ax.set_ylabel("...")` |
| `plt.xlim(0, 1)` | `ax.set_xlim(0, 1)` |
| `plt.plot`, `plt.legend`, `plt.grid` | `ax.plot`, `ax.legend`, `ax.grid` (the same names) |

With 2 rows and 2 columns, `axes` is a grid and you write `axes[row, column]`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(sw_full, kro_full, color="tab:green", label="$k_{ro}$")
axes[0].plot(sw_full, krw_full, color="tab:blue", label="$k_{rw}$")
axes[0].set_title("Relative permeability")
axes[0].set_xlabel("water saturation $S_w$ (fraction)")
axes[0].set_ylabel("relative permeability (fraction)")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(sw_c, fw_c, color="tab:red")
axes[1].set_title("Fractional flow")
axes[1].set_xlabel("water saturation $S_w$ (fraction)")
axes[1].set_ylabel("fractional flow $f_w$ (fraction)")
axes[1].grid(True)

plt.tight_layout()
plt.show()

**Your turn 6.1.** Make a figure with **three** panels in a row for the **laboratory table**: (left) $k_{ro}$ and $k_{rw}$ against `sw_t`, (middle) $f_w$ (`fw_t`) against `sw_t`, (right) the capillary pressure `pc_t` against `sw_t`. Give every panel a title and axis labels (with units for pressure).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# YOUR CODE HERE
# draw on axes[0], axes[1], axes[2]

plt.tight_layout()
plt.show()

In [ ]:
# Check (do not edit)
assert len(axes) == 3, "the figure should have three panels"
assert len(axes[0].lines) >= 2, "left panel: two curves (kro and krw)"
assert len(axes[1].lines) >= 1 and len(axes[2].lines) >= 1, "the middle and right panels need a curve each"
for ax in axes:
    assert ax.get_title() and ax.get_xlabel() and ax.get_ylabel(), "every panel needs a title and both axis labels"
print("Correct")

<details><summary>Solution</summary>

```python
axes[0].plot(sw_t, kro_t, "o-", color="tab:green", markersize=3, label="$k_{ro}$")
axes[0].plot(sw_t, krw_t, "o-", color="tab:blue", markersize=3, label="$k_{rw}$")
axes[0].set_title("Relative permeability")
axes[0].set_xlabel("water saturation $S_w$ (fraction)")
axes[0].set_ylabel("relative permeability (fraction)")
axes[0].legend()

axes[1].plot(sw_t, fw_t, "o-", color="tab:red", markersize=3)
axes[1].set_title("Fractional flow")
axes[1].set_xlabel("water saturation $S_w$ (fraction)")
axes[1].set_ylabel("fractional flow $f_w$ (fraction)")

axes[2].plot(sw_t, pc_t, "o-", color="tab:purple", markersize=3)
axes[2].set_title("Capillary pressure")
axes[2].set_xlabel("water saturation $S_w$ (fraction)")
axes[2].set_ylabel("$P_{cow}$ (psi)")

for ax in axes:
    ax.grid(True)
plt.tight_layout()
plt.show()
```

</details>

## 7. Predict, then run: a more viscous oil

The oil viscosity $\mu_o$ appears in equation (1). Water (1 cp) will displace three different oils: a light one (0.5 cp), our medium one (2 cp), and a heavy one (10 cp).

**Predict** (write your answers in the cell below before you run anything):

1. For the heavy oil, does the $f_w$ curve move to the **left** or to the **right** of the medium oil curve?
2. For the heavy oil, at a given saturation, is the flow *more* or *less* water than before? Why?
3. Is that good or bad news for the waterflood?

In [ ]:
# 1.
# 2.
# 3.

In [ ]:
sw_v = np.linspace(0.2, 0.8, 601)
krw_v, kro_v = corey_krw_kro(sw_v, corey)

plt.figure(figsize=(6, 4))
for mu_o, colour in [(0.5, "tab:green"), (2.0, "tab:blue"), (10.0, "tab:red")]:
    fw_v = fractional_flow(krw_v, kro_v, MU_W, mu_o)
    plt.plot(sw_v, fw_v, color=colour, linewidth=2, label=f"$\\mu_o$ = {mu_o} cp")
plt.title("Fractional flow for three oils")
plt.xlabel("water saturation $S_w$ (fraction)")
plt.ylabel("fractional flow $f_w$ (fraction)")
plt.legend()
plt.grid(True)
plt.show()

The more viscous the oil, the further **left** the curve moves. The heavy oil is hard to move, so at the same saturation the water takes a larger share of the flow, and water reaches the producer earlier and in larger amounts. The viscosity enters through the mobility ratio you met on Day 7, and a high ratio means a poor sweep.

**Your turn 7.1.** For each oil viscosity in `[0.5, 2.0, 10.0]` find the saturation where $f_w$ first reaches 0.5 (as in 5.2, on the 601-point arrays `sw_v`, `krw_v`, `kro_v`). Store the three saturations in a dictionary `sw_half_by_mu` with the viscosity as the key (Day 6!).

In [ ]:
# YOUR CODE HERE
sw_half_by_mu = {}

In [ ]:
# Check (do not edit)
assert set(sw_half_by_mu) == {0.5, 2.0, 10.0}, "the keys should be the three viscosities"
assert abs(sw_half_by_mu[0.5] - 0.6146) < 0.003, f"0.5 cp: expected about 0.615, got {sw_half_by_mu[0.5]}"
assert abs(sw_half_by_mu[2.0] - 0.5167) < 0.003, f"2 cp: expected about 0.517, got {sw_half_by_mu[2.0]}"
assert abs(sw_half_by_mu[10.0] - 0.4) < 0.003, f"10 cp: expected about 0.400, got {sw_half_by_mu[10.0]}"
print("Correct:", {k: round(float(v), 3) for k, v in sw_half_by_mu.items()})

<details><summary>Solution</summary>

```python
sw_half_by_mu = {}
for mu_o in [0.5, 2.0, 10.0]:
    fw_v = fractional_flow(krw_v, kro_v, MU_W, mu_o)
    sw_half_by_mu[mu_o] = sw_v[fw_v >= 0.5][0]
```
The 50 % point drops from 0.61 to 0.40 as the oil gets 20 times more viscous: the heavy oil needs much less water in the rock before the produced fluid is half water.
</details>

## 8. A first look ahead: the water front

On Day 15 you will meet the Buckley-Leverett solution fully. Today we use only the idea and the tools. Suppose water is injected into a bed that holds only connate water, $S_{wi}=S_{wc}=0.2$. The water does not spread out smoothly: a **sharp front** travels through the bed. Welge's construction says that the saturation just behind the front is where the straight line from the starting point $(S_{wi},\,f_w(S_{wi}))$ **touches** the $f_w$ curve from above. In other words, it is where the slope

$$\frac{f_w(S_w)-f_w(S_{wi})}{S_w-S_{wi}}$$

is the **largest**. The reciprocal of that largest slope is the pore volumes of water injected when water breaks through at the producer.

This is a good use of a mask: the slope is only defined for $S_w > S_{wi}$ (otherwise we divide by zero).

**Your turn 8.1.** Write `front_saturation(sw, fw, swi)` returning `(sw_front, max_slope)`: select the points with `sw > swi`, calculate the slope for them (use `fw[0]` as the value at the starting point; the arrays start at `swi`), and find the saturation where the slope is largest (`argmax` from Day 7). Then store in `breakthrough_pv` the value $1/\text{max slope}$.

In [ ]:
fw_v = fractional_flow(krw_v, kro_v, MU_W, 2.0)

# YOUR CODE HERE
def front_saturation(sw, fw, swi):
    """Saturation at the shock front and the largest slope of the chord from (swi, fw[0])."""
    pass

breakthrough_pv = None

In [ ]:
# Check (do not edit)
sw_front, max_slope = front_saturation(sw_v, fw_v, 0.2)
assert abs(sw_front - 0.647) < 0.005, f"the front saturation should be about 0.647, got {sw_front}"
assert abs(max_slope - 1.951) < 0.01, f"the largest slope should be about 1.95, got {max_slope}"
assert abs(breakthrough_pv - 0.5125) < 0.005, f"breakthrough after about 0.51 pore volumes, got {breakthrough_pv}"
print(f"Correct: front at Sw = {sw_front:.3f}; water breaks through after {breakthrough_pv:.2f} pore volumes")

<details><summary>Solution</summary>

```python
def front_saturation(sw, fw, swi):
    """Saturation at the shock front and the largest slope of the chord from (swi, fw[0])."""
    ahead = sw > swi
    slope = (fw[ahead] - fw[0]) / (sw[ahead] - swi)
    i = slope.argmax()
    return sw[ahead][i], slope[i]

sw_front, max_slope = front_saturation(sw_v, fw_v, 0.2)
breakthrough_pv = 1 / max_slope
```
With the 2 cp oil, water arrives at the producer after about half a pore volume has been injected. Try the 10 cp oil: the answer is about 0.37, i.e. earlier.
</details>

In [ ]:
# The picture: the chord from (swi, fw(swi)) touching the curve
plt.figure(figsize=(6, 4))
plt.plot(sw_v, fw_v, color="tab:blue", linewidth=2, label="$f_w$")
plt.plot([0.2, sw_front], [fw_v[0], np.interp(sw_front, sw_v, fw_v)], "k--", label="tangent from $S_{wi}$")
plt.plot(sw_front, np.interp(sw_front, sw_v, fw_v), "ko")
plt.title(f"Shock front at $S_w$ = {sw_front:.2f}")
plt.xlabel("water saturation $S_w$ (fraction)")
plt.ylabel("fractional flow $f_w$ (fraction)")
plt.legend()
plt.grid(True)
plt.show()

## 9. Bridge: these tools inside `kusim`

Look for the same ideas in the package, by name (use your editor's search):

- **`np.clip` for saturation limits.** In `src/kusim/simulator.py` the explicit saturation update ends with `s_new = np.clip(s_new, m.swc, 1.0 - m.sor)`. That is your Section 4: after each time step, saturations that overshoot the physical limits are put back.
- **Fractional flow.** `src/kusim/analytical.py` has `fractional_flow(rock_fluid, sw, mu_o, mu_w)`. It is equation (1) in its second form, so it has to guard against $k_{rw}=0$. It does so with `np.where(krw > 0, kro * mu_w / (krw * mu_o), np.inf)`: "where the water can flow, use the ratio; elsewhere use infinity" (and $1/(1+\infty)=0$). (Note the order of its arguments: `mu_o` comes before `mu_w`, unlike our function today.)
- **Reading arrays backwards.** The same file uses `x_of_sw[::-1]` before `np.interp`, and `tables.py` has `invert_decreasing`, your `sw_from_pc`.
- **A `np.where` that chooses the upstream block.** In `simulator.py`, the mobility between two blocks is taken from the block the fluid flows *from*: `np.where(self._neighbour_is_upstream(p_old, m.gamma_o, head[d], nb), pr.lam_o[nb], pr.lam_o)` reads "where the neighbour is upstream, use its mobility, otherwise use this block's". This is called **upstream weighting**, and we meet it on Day 11. The test `_neighbour_is_upstream` compares the *potential* (the pressure corrected for the weight of the fluid column between the blocks); for two blocks at the same depth that is simply "the neighbour has the higher pressure", which is how the thesis states the rule. The correction matters when fluid flows up or down, and the bonus project notebook (`course/bonus/`) shows why.

## 10. Self-quiz

Write short answers as comments, then open the answers.

1. `a = np.array([10, 20, 30, 40, 50])`: what are `a[1:3]`, `a[-2:]`, `a[::-1]` and `a[a > 25]`?
2. Why do we write `(a > 1) & (a < 5)` and not `(a > 1) and (a < 5)`?
3. What is the difference between `b = a[1:3]` and `b = a[1:3].copy()` when you change `b`?
4. `np.clip(x, 0, 1)` and `np.where(x > 1, 1, np.where(x < 0, 0, x))` give the same result. Which would you prefer to read, and why?
5. At a producer, $f_w = 0.8$. What does that mean? And why does a heavier oil make the water cut rise earlier?

In [ ]:
# 1.
# 2.
# 3.
# 4.
# 5.

<details><summary>Answers</summary>

1. `[20 30]`, `[40 50]`, `[50 40 30 20 10]` and `[30 40 50]` (the items greater than 25).
2. `and` asks for *one* True/False for the whole array, which is ambiguous (NumPy raises a `ValueError`). `&` works item by item, and the parentheses are needed because `&` binds more tightly than `>`.
3. Without `.copy()`, `b` is a window onto `a`: changing `b` changes `a` as well. With `.copy()` the two are separate.
4. `np.clip(x, 0, 1)`: it is shorter and says what it does, with no nested conditions to read.
5. Eighty percent of the fluid flowing at that point is water (and 20 % oil). A heavier oil has a lower mobility $\lambda_o$, so in $f_w = \lambda_w/(\lambda_w+\lambda_o)$ the water gets a bigger share at the same saturation, and the water cut rises earlier.
</details>

## 11. Pocket reference (today's Python)

| You want to... | Write |
|---|---|
| items 2, 3, 4 | `a[2:5]` |
| first / last `n` items | `a[:n]`, `a[-n:]` |
| every `k`-th item | `a[::k]` |
| backwards | `a[::-1]` |
| difference of neighbours | `a[1:] - a[:-1]` |
| a separate copy | `a[1:3].copy()` |
| a mask | `mask = a > 5` |
| select by a mask | `a[mask]` |
| count / any / all | `mask.sum()`, `mask.any()`, `mask.all()` |
| and / or / not (masks) | `(a > 1) & (a < 5)`, `(a < 1) \| (a > 5)`, `~mask` |
| replace selected items | `a[a > 0.8] = 0.8` |
| compare decimals | `np.isclose(a, 0.3)` |
| an `if` for every item | `np.where(cond, if_true, if_false)` |
| limit to a range | `np.clip(a, low, high)` |
| several panels | `fig, axes = plt.subplots(1, 3, figsize=(14, 4))`, `axes[0].plot(...)`, `ax.set_title(...)`, `plt.tight_layout()` |

## 12. Wrap-up

Today you:

- sliced arrays, read them backwards, and met the shifted-slice trick for differences,
- selected, counted and replaced items with boolean masks,
- limited saturations to their physical range with `np.clip`, and fixed the Corey curves outside their limits,
- derived and computed the fractional flow of water, and saw how oil viscosity moves the curve,
- used a mask to find the shock front, a first taste of Buckley-Leverett,
- drew multi-panel figures,
- found the same tools in `kusim`.

**Next, Day 9:** Darcy's law in one dimension, with its conversion factor 0.001127, and the pressure gradient as a finite difference: your shifted slices, now called `np.diff`.